**Author:** Javier Lendínez Castillo  
**License:** GNU Affero General Public License v3.0 ([LICENSE](../../LICENSE))

# Geometry, Discretization, and Control Surfaces

In this notebook, I build a tapered wing with sweep, camber, and a flap using the actual `VLM` and `geometry` APIs. I inspect the panel geometry, compare its projections, and analyze how the solution changes with flap deflection.

The example extends the geometry cases covered in my thesis. I retain the method's assumptions: steady, incompressible, inviscid flow over a thin lifting surface, without a separation or viscous-drag model.

In [63]:
import sys
from pathlib import Path


def _find_repo_root(start: Path) -> Path:
    current = start
    while current != current.parent:
        if (current / 'src' / 'vlm').is_dir():
            return current
        current = current.parent
    raise RuntimeError('Could not locate the VLMPy repository root (expected src/vlm).')


REPO_ROOT = _find_repo_root(Path.cwd())
SRC_VLM = REPO_ROOT / 'src' / 'vlm'
if str(SRC_VLM) not in sys.path:
    sys.path.insert(0, str(SRC_VLM))

print(f'VLM source path added to sys.path: {SRC_VLM}')

VLM source path added to sys.path: c:\Users\javil\OneDrive - Universidad Alfonso X el Sabio\Documents\GitHub\VLMPy\src\vlm


In [64]:
import contextlib
import io

import numpy as np
import plotly.graph_objects as go

from lib.vlm import VLM
from lib.geometry import (
    plot_wing_geometry_2d,
    plot_wing_discretization_2d,
    plot_wing_discretization_3d,
)

## 1. Tapered Wing with Camber and Flap

In this configuration, `sweep` and `dihedral` are specified in radians; `flap_start` and `flap_end` delimit fractions of the semispan; and `flap_hinge_chord` gives the hinge location as a fraction of the local chord.

For `deflection_type='symmetrical'`, positive deflection moves the trailing edge downward, according to the geometric transformation in `generate_panels_for_surface`. I interpret the lift increase as a trend for this case and this kinematic model, not as a prediction of separation or maximum flap load.

In [65]:
plane = {
    'wing_sections': [
        {
            'chord_root': 1.2,
            'chord_tip': 0.6,
            'span_fraction': 8.0,          # total span [m]
            'sweep': np.radians(20.0),     # leading-edge sweep [rad]
            'dihedral': np.radians(4.0),
            'NACA_root': '4412',
            'NACA_tip': '2412',
            'flap_start': 0.55,            # fraction of half-span
            'flap_end': 0.90,
            'flap_hinge_chord': 0.75,       # hinge at 75% local chord
            'deflection_angle': np.radians(15.0),
            'deflection_type': 'symmetrical',
        }
    ]
}

vlm = VLM(plane=plane, u=45.0, rho=1.225, alpha=np.radians(4.0), beta=0.0, n=24, m=8)

## 2. Geometry and Discretization by Stage

I run geometry calculation and panel generation separately to inspect the cost and cache indicators for each stage. This separation also helps identify which stage first shows an inconsistency between geometry and mesh.

In [66]:
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm.calculate_geometry()
    vlm.calculate_discretization()

print(f'Wing span (this section): {vlm.total_wing_span:.2f} m')
print(f'Panels: n={vlm.n}, m={vlm.m}, total wing panels={vlm.n * vlm.m}')
print(f'Wing planform area: {vlm.wing_area:.4f} m^2')
print('Timings [s]:', {k: v for k, v in vlm.timings.items() if not k.endswith('_hit')})
print('Cache-hit flags:', {k: v for k, v in vlm.timings.items() if k.endswith('_hit')})

Wing span (this section): 8.00 m
Panels: n=24, m=8, total wing panels=192
Wing planform area: 7.2302 m^2
Timings [s]: {'geometry': 0.00015849992632865906, 'discretization': 0.0207632000092417}
Cache-hit flags: {'geometry_cache_hit': True, 'discretization_cache_hit': False}


## 3. Panel Structure

Each element of `vlm.panel_data` contains the index, corners, quarter-chord line nodes, control point, and auxiliary normal data. Coordinates follow the internal `(y, x, z)` convention: spanwise, chordwise, and vertical. Inspecting this record connects solver quantities to the geometry being plotted.

In [67]:
index, corners, quarter_chord_nodes, control_point, normal_info = vlm.panel_data[0]
normal_vector, three_quarter_midpoint = normal_info

print(f'Panel index: {index}')
print('Corner coordinates (y, x, z):')
for corner in corners:
    print(f'  {corner}')
print(f'Quarter-chord (bound vortex) nodes: {quarter_chord_nodes}')
print(f'Control point (3/4 chord): {control_point}')
print(f'Panel normal vector: {normal_vector}')

Panel index: 1
Corner coordinates (y, x, z):
  [-3.9902562   1.46826053  0.17703647]
  [-3.9902562   1.54307784  0.17180473]
  [-3.65773485  1.42695757  0.15661572]
  [-3.65773485  1.34590549  0.16228343]
Quarter-chord (bound vortex) nodes: [[-3.990256201039297, 1.4869648589240536, 0.17572853571745084], [-3.657734850952689, 1.3661685083374444, 0.16086650209387954]]
Control point (3/4 chord): [-3.8239955259959926, 1.4655340293440235, 0.1655726566500353]
Panel normal vector: [0.06975647 0.06958655 0.99513403]


## 4. Camber Slope $dz/dx$

`vlm.dz_c` stores the local slope associated with the panels and contributes to the boundary condition. I check that there is one entry per panel and summarize its range. This check verifies the correspondence between geometry data and panels; it is not, by itself, a comparison with an experimental airfoil profile.

In [68]:
print(f'dz/dx entries: {len(vlm.dz_c)} (panel count: {len(vlm.panel_data)})')
assert len(vlm.dz_c) == len(vlm.panel_data)
print(f'Camber slope range: [{min(vlm.dz_c):.4f}, {max(vlm.dz_c):.4f}]')

dz/dx entries: 192 (panel count: 192)
Camber slope range: [-0.1687, 0.1194]


## 5. Geometry and Mesh Views

The three views distinguish the planform, side projection, and spatial panel layout. The 2D view helps inspect edges and quarter-chord lines; the 3D view can reveal orientation errors that may be missed in a flat projection.

In [69]:
fig_planform = go.Figure()
fig_planform = plot_wing_geometry_2d(fig_planform, vlm.wing_geometry, legend='Wing Geometry')
fig_planform.show()

In [70]:
fig_panels = go.Figure()
fig_panels = plot_wing_discretization_2d(fig_panels, vlm.panel_data)
fig_panels.show()

Execution time: 0.04 seconds


In [71]:
fig_3d = go.Figure()
fig_3d = plot_wing_discretization_3d(fig_3d, vlm.panel_data)
fig_3d.show()

Execution time: 0.03 seconds


## 5. Geometry Views and Mesh Inspection

The planform, side, and 3D views make it possible to inspect the panel arrangement from different directions. Reviewing all three helps catch geometric details that are difficult to see in a single projection.

In [72]:
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm.run_vlm()
cl_deflected = vlm.CL
cd_deflected = vlm.CD

plane_neutral = {'wing_sections': [dict(plane['wing_sections'][0])]}
plane_neutral['wing_sections'][0]['deflection_angle'] = 0.0

vlm_neutral = VLM(plane=plane_neutral, u=45.0, rho=1.225, alpha=np.radians(4.0), beta=0.0, n=24, m=8)
_out = io.StringIO()
with contextlib.redirect_stdout(_out):
    vlm_neutral.run_vlm()

deflection_deg = np.degrees(plane['wing_sections'][0]['deflection_angle'])
print(f'Flap neutral   : CL={vlm_neutral.CL:.4f}, CD={vlm_neutral.CD:.5f}')
print(f'Flap deflected : CL={cl_deflected:.4f}, CD={cd_deflected:.5f}')
print(f'Delta CL from a {deflection_deg:.0f} deg deflection: {cl_deflected - vlm_neutral.CL:+.4f}')

assert cl_deflected > vlm_neutral.CL, 'A positive (trailing-edge-down) flap deflection should increase CL at fixed alpha.'

Flap neutral   : CL=0.6168, CD=0.00840
Flap deflected : CL=0.8280, CD=0.02213
Delta CL from a 15 deg deflection: +0.2112


## 7. Limitations

- The flap is represented by a rigid transformation of points downstream of the hinge; hinge moment and a physical discontinuity between surfaces are not modeled.
- The model does not include viscosity, separation, or a deflection limit associated with loss of flap effectiveness. Therefore, results at 20° describe only the mathematical response of the linear VLM.
- The response depends on discretization; quantifying sensitivity would require repeating the sweep on more than one mesh.

## 8. Flap-Deflection Sweep

I keep the wing, angle of attack, and flight conditions fixed. I vary only the flap deflection and plot $C_L$ and $C_D$ separately so their different scales do not hide either trend. The cell also checks that all cases retain the effective mesh `(12, 6)`.

On this mesh, increasing deflection from 0° to 20° raises $C_L$ from 0.61390 to 0.87112 and induced $C_D$ from 0.008078 to 0.025491. The trend is monotonic over this interval. At 15°, the `(12, 6)` mesh sweep gives $C_L=0.80813$, compared with 0.8280 in the neutral/deflected comparison using `(24, 8)`. This difference is not a physical contradiction: the calculations use different resolutions.

In [77]:
import contextlib
import io
import sys
from pathlib import Path

import numpy as np
import plotly.graph_objects as go

current = Path.cwd()
while current != current.parent and not (current / 'src' / 'vlm').is_dir():
    current = current.parent
if not (current / 'src' / 'vlm').is_dir():
    raise RuntimeError('Could not locate the VLMPy repository root.')
sys.path.insert(0, str(current / 'src' / 'vlm'))

from lib.vlm import VLM

section_sweep = {
    'chord_root': 1.2, 'chord_tip': 0.6, 'span_fraction': 8.0,
    'sweep': np.radians(20.0), 'dihedral': np.radians(4.0),
    'NACA_root': '4412', 'NACA_tip': '2412',
    'flap_start': 0.55, 'flap_end': 0.90,
    'flap_hinge_chord': 0.75, 'deflection_type': 'symmetrical',
}
deflections_deg = [0.0, 5.0, 10.0, 15.0, 20.0]
cl_by_deflection = []
cd_by_deflection = []

for deflection_deg in deflections_deg:
    section_case = dict(section_sweep)
    section_case['deflection_angle'] = np.radians(deflection_deg)
    plane_case = {'wing_sections': [section_case]}
    case = VLM(plane_case, 45.0, 1.225, np.radians(4.0), 0.0, 12, 6)
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (12, 6), 'The solver changed the requested mesh.'
    cl_by_deflection.append(float(case.CL))
    cd_by_deflection.append(float(case.CD))
    print(
        'Deflection={:>4.0f}°; effective mesh=({},{}): CL={:.5f}, CD={:.6f}'.format(
            deflection_deg, case.n, case.m, case.CL, case.CD
        )
    )

cl_figure = go.Figure()
cl_figure.add_trace(go.Scatter(
    x=deflections_deg, y=cl_by_deflection,
    mode='lines+markers', name='CL',
))
cl_figure.update_layout(
    title='Effect of flap deflection on CL',
    xaxis_title='Flap deflection [deg]',
    yaxis_title='CL [-]', template='plotly_white',
)
cl_figure.show()

cd_figure = go.Figure()
cd_figure.add_trace(go.Scatter(
    x=deflections_deg, y=cd_by_deflection,
    mode='lines+markers', name='Induced CD',
))
cd_figure.update_layout(
    title='Effect of flap deflection on induced CD',
    xaxis_title='Flap deflection [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
)
cd_figure.show()

assert np.isfinite(cl_by_deflection).all()
assert np.isfinite(cd_by_deflection).all()

Deflection=   0°; effective mesh=(12,6): CL=0.61390, CD=0.008078
Deflection=   5°; effective mesh=(12,6): CL=0.67935, CD=0.010804
Deflection=  10°; effective mesh=(12,6): CL=0.74416, CD=0.014674
Deflection=  15°; effective mesh=(12,6): CL=0.80813, CD=0.019604
Deflection=  20°; effective mesh=(12,6): CL=0.87112, CD=0.025491


## 6. Neutral and Deflected Flap Comparison

I solve the same geometry and flight conditions with the flap deflected and with zero deflection. The difference in $C_L$ quantifies the model response for this case; it should not be attributed to viscous effects, which are not included.

In [78]:
aileron_section = {
    'chord_root': 1.0, 'chord_tip': 1.0, 'span_fraction': 10.0,
    'sweep': 0.0, 'dihedral': 0.0,
    'NACA_root': '0106', 'NACA_tip': '0106',
    'flap_start': 0.5, 'flap_end': 1.0,
    'flap_hinge_chord': 0.75, 'deflection_type': 'antisymmetrical',
}
aileron_deflections_deg = [-12.0, 0.0, 12.0]
aileron_cl_global = []
aileron_cd_induced = []
aileron_cl_local = []
aileron_y = None

for deflection_deg in aileron_deflections_deg:
    section_case = dict(aileron_section)
    section_case['deflection_angle'] = np.radians(deflection_deg)
    case = VLM(
        {'wing_sections': [section_case]}, 50.0, 1.225,
        0.0, 0.0, 24, 8,
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (24, 8)
    y_positions = np.array([
        np.mean([
            panel[3][0]
            for panel in case.panel_data[i * case.m:(i + 1) * case.m]
        ])
        for i in range(case.n)
    ])
    if aileron_y is None:
        aileron_y = y_positions
    else:
        np.testing.assert_allclose(y_positions, aileron_y, atol=1e-12)
    aileron_cl_global.append(float(case.CL))
    aileron_cd_induced.append(float(case.CD))
    aileron_cl_local.append(np.asarray(case.CL_locals_wing, dtype=float))
    left_lift = float(np.sum(np.asarray(case.lift_wing)[y_positions < 0.0]))
    right_lift = float(np.sum(np.asarray(case.lift_wing)[y_positions > 0.0]))
    print(
        'Deflection={:+.0f}°: CL={:.6f}, induced CD={:.6f}, left/right lift={:.2f}/{:.2f} N'.format(
            deflection_deg, case.CL, case.CD, left_lift, right_lift
        )
    )

zero_index = aileron_deflections_deg.index(0.0)
positive_index = aileron_deflections_deg.index(12.0)
negative_index = aileron_deflections_deg.index(-12.0)
assert abs(aileron_cl_global[zero_index]) < 1e-8
assert aileron_cd_induced[positive_index] > aileron_cd_induced[zero_index]
np.testing.assert_allclose(
    aileron_cl_local[positive_index], aileron_cl_local[negative_index][::-1],
    rtol=2e-5, atol=5e-6,
 )
np.testing.assert_allclose(
    aileron_cd_induced[positive_index], aileron_cd_induced[negative_index],
    rtol=1e-8, atol=1e-10,
 )

aileron_figure = go.Figure()
for deflection_deg, local_cl in zip(aileron_deflections_deg, aileron_cl_local):
    aileron_figure.add_trace(go.Scatter(
        x=aileron_y, y=local_cl, mode='lines+markers',
        name='Deflection={:+.0f}°'.format(deflection_deg),
    ))
aileron_figure.update_layout(
    title='Spanwise CL distribution with antisymmetric ailerons',
    xaxis_title='Spanwise coordinate y [m]',
    yaxis_title='Local CL [-]', template='plotly_white',
 )
aileron_figure.show()

aileron_cd_figure = go.Figure()
aileron_cd_figure.add_trace(go.Scatter(
    x=aileron_deflections_deg, y=aileron_cd_induced,
    mode='lines+markers', name='Induced CD',
))
aileron_cd_figure.update_layout(
    title='Induced CD versus antisymmetric deflection',
    xaxis_title='Deflection on the +y semispan [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
 )
aileron_cd_figure.show()

assert np.isfinite(aileron_cl_global).all()
assert np.isfinite(aileron_cd_induced).all()

Deflection=-12°: CL=0.000000, induced CD=0.009964, left/right lift=2100.96/-2100.96 N
Deflection=+0°: CL=0.000000, induced CD=0.000000, left/right lift=0.00/0.00 N
Deflection=+12°: CL=-0.000000, induced CD=0.009964, left/right lift=-2100.96/2100.96 N


## 10. Winglets: Comparison with Thesis Section 6.7

I reproduce the baseline setup from Section 6.5: a symmetric wing with a 10 m span, 1 m chord, NACA 4412, 30° sweep, 5° dihedral, $\alpha=0^\circ$, $V=50\,\mathrm{m/s}$, and $\rho=1.225\,\mathrm{kg/m^3}$. The winglet has 0.10/0.05 m root/tip chords and NACA 4412/0012 sections; I sweep its dihedral from 5° to 75° on a 100×10 mesh. In the direct API, `span_fraction` is used as a physical section length; I use 1.0 m so the vertical height at 75° is $1\sin(75^\circ)\approx0.483$ m. This differs from the `0.1` described as a fraction in the thesis interface, so the convention should be clarified in the application/documentation.

The thesis attributes a reduction in induced drag to winglets. This calculation tests that claim rather than assuming it: I compare $C_{D_i}$ against the baseline wing and plot the results for each dihedral angle. The observed discrepancy is not, by itself, a causal diagnosis or model validation; the VLM idealization and geometry definition should be reviewed before drawing a design conclusion.

In [79]:
winglet_base_section = {
    'chord_root': 1.0, 'chord_tip': 1.0, 'span_fraction': 10.0,
    'sweep': np.radians(30.0), 'dihedral': np.radians(5.0),
    'NACA_root': '4412', 'NACA_tip': '4412',
}
winglet_base_conditions = (50.0, 1.225, 0.0, 0.0, 100, 10)
winglet_cant_angles_deg = np.array([5, 15, 25, 35, 45, 55, 65, 75])
winglet_cl = []
winglet_cdi = []

base_case = VLM(
    {'wing_sections': [winglet_base_section]}, *winglet_base_conditions,
 )
captured = io.StringIO()
with contextlib.redirect_stdout(captured):
    base_case.run_vlm()
base_cl = float(base_case.CL)
base_cdi = float(base_case.CD)
print('Baseline wing: CL={:.6f}, CDi={:.6f}'.format(base_cl, base_cdi))

for cant_deg in winglet_cant_angles_deg:
    winglet_section = {
        'chord_root': 0.1, 'chord_tip': 0.05,
        'span_fraction': 1.0, 'sweep': 0.0,
        'dihedral': np.radians(float(cant_deg)),
        'NACA_root': '4412', 'NACA_tip': '0012',
    }
    case = VLM(
        {'wing_sections': [winglet_base_section, winglet_section]},
        *winglet_base_conditions,
    )
    captured = io.StringIO()
    with contextlib.redirect_stdout(captured):
        case.run_vlm()
    assert (case.n, case.m) == (100, 10)
    assert np.isfinite(case.gammas).all()
    winglet_cl.append(float(case.CL))
    winglet_cdi.append(float(case.CD))
    vertical_height = 0.5 * np.sin(np.radians(float(cant_deg)))
    print(
        'Cant angle={:2.0f}°; vertical height per tip={:.3f} m; CL={:.6f}; CDi={:.6f}'.format(
            cant_deg, vertical_height, case.CL, case.CD
        )
    )

assert np.isfinite(winglet_cl).all()
assert np.isfinite(winglet_cdi).all()
winglet_cl_figure = go.Figure()
winglet_cl_figure.add_trace(go.Scatter(
    x=winglet_cant_angles_deg, y=winglet_cl,
    mode='lines+markers', name='With winglets',
))
winglet_cl_figure.add_hline(
    y=base_cl, line_dash='dash', annotation_text='Baseline wing',
 )
winglet_cl_figure.update_layout(
    title='CL versus winglet cant angle',
    xaxis_title='Winglet cant angle [deg]',
    yaxis_title='CL [-]', template='plotly_white',
 )
winglet_cl_figure.show()

winglet_cdi_figure = go.Figure()
winglet_cdi_figure.add_trace(go.Scatter(
    x=winglet_cant_angles_deg, y=winglet_cdi,
    mode='lines+markers', name='With winglets',
))
winglet_cdi_figure.add_hline(
    y=base_cdi, line_dash='dash', annotation_text='Baseline wing',
 )
winglet_cdi_figure.update_layout(
    title='Induced CD versus winglet cant angle',
    xaxis_title='Winglet cant angle [deg]',
    yaxis_title='Induced CD [-]', template='plotly_white',
 )
winglet_cdi_figure.show()

print(
    'Change in CDi versus baseline wing: min={:+.1f} %, max={:+.1f} %'.format(
        100.0 * (np.min(winglet_cdi) / base_cdi - 1.0),
        100.0 * (np.max(winglet_cdi) / base_cdi - 1.0),
    )
 )

Baseline wing: CL=0.333772, CDi=0.002432
Cant angle= 5°; vertical height per tip=0.044 m; CL=0.339212; CDi=0.002925
Cant angle=15°; vertical height per tip=0.129 m; CL=0.338749; CDi=0.002878
Cant angle=25°; vertical height per tip=0.211 m; CL=0.339851; CDi=0.002987
Cant angle=35°; vertical height per tip=0.287 m; CL=0.338749; CDi=0.002893
Cant angle=45°; vertical height per tip=0.354 m; CL=0.336505; CDi=0.002714
Cant angle=55°; vertical height per tip=0.410 m; CL=0.336485; CDi=0.002744
Cant angle=65°; vertical height per tip=0.453 m; CL=0.335046; CDi=0.002638
Cant angle=75°; vertical height per tip=0.483 m; CL=0.331005; CDi=0.002502


Change in CDi versus baseline wing: min=+2.9 %, max=+22.8 %


### Results and Limitations

With the stated mesh, the baseline wing gives $C_L=0.333772$ and $C_{D_i}=0.002432$. Across the eight tested dihedral angles, $C_L$ ranges from 0.331005 to 0.339851, while $C_{D_i}$ ranges from 0.002502 to 0.002987: it increases in every case relative to the baseline, by 2.9% to 22.8%. Therefore, this run does not reproduce the induced-drag reduction attributed to winglets in the thesis. The trend is not monotonic with dihedral either.

This comparison identifies follow-up work, not which hypothesis explains the difference: review the `span_fraction` convention, the exact original study inputs, and the discretization/camber calculation. In exploratory runs, `n=100` solved the full sweep; a coarse mesh failed during camber calculation at high dihedral because the routine assigned no points to the short segment. This remains a documented code limitation; it is not corrected or hidden here. VLM also does not model viscous drag or optimize real winglets.